# Ask Alfonso · training on a GPU (v6)

v6 goes back to v4's model (4 layers, 256-dim, ~4M parameters — it generalized better than
v5's bigger one) and changes the **data**: a paraphrase grammar gives hundreds of ways to ask
each question, and every epoch the model sees fresh ones (`ml/askalfonso/paraphrase.py`).
Cell 9 compares v4 and v6 on dev **and** on the stress set (453 questions in ~85 phrasings no
model has trained on), which is the better guide to how it handles real visitors.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
Cell 2 asks for permission to use your Google Drive: progress is saved there.

**If Colab disconnects:** reconnect and `Run all` again — it continues where it stopped.
If the output stops moving but the stop button still spins, it's usually still running:
check `MyDrive/ask-alfonso/checkpoints/colab-v6.log`.


In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Settings, and Google Drive for checkpoints (survives disconnects)
RUN = "colab-v6"
ARGS = "--paraphrases 6"   # 6 fresh grammar phrasings per (question type, project/company) every epoch

import os
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/ask-alfonso/checkpoints"
os.makedirs(CKPT, exist_ok=True)
print("Checkpoints and log →", f"{CKPT}/{RUN}")


In [ ]:
# 3. Get the code (re-running this cell pulls the latest version)
import os
REPO = "https://github.com/Alfonso3435/portfolio.git"
if not os.path.exists("/content/portfolio"):
    !git clone -q {REPO} /content/portfolio
%cd /content/portfolio
!git pull -q
!git log -1 --format="Commit %h · %s"

In [ ]:
# 4. Phase 1 — export the site data and build the dataset (needs Node.js for the TypeScript export)
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y -qq nodejs)
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!npm run --silent export:knowledge
!python ml/build_dataset.py --samples 0

In [ ]:
# 5. Phase 2 — train the tokenizer (deterministic: identical to the one on your laptop)
!python ml/train_tokenizer.py | tail -8

### Pretraining on general English (stage A)

Same as v4: 3,000 steps of next-token prediction on WikiText-2 before the Q&A task. The text
and its tokens are cached in Drive.


In [ ]:
# 6. Download WikiText-2 (once; cached in Drive)
WIKI = f"{CKPT}/wikitext2.txt"
if not os.path.exists(WIKI):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    with open(WIKI + ".tmp", "w", encoding="utf-8") as f:
        f.write("\n".join(t for t in wiki["text"] if t.strip()))
    os.replace(WIKI + ".tmp", WIKI)
PRETRAIN_ARGS = f"--pretrain-text {WIKI} --pretrain-steps 3000 --pretrain-cache {CKPT}/{RUN}/wiki-tokens.pt"
print(f"{os.path.getsize(WIKI) / 1e6:.0f} MB of pretraining text ready")


In [ ]:
# 7. Train (stage A, then stage B with fresh paraphrases every epoch). Resumes automatically.
#    Every 4 epochs: dev EM (exactly right), F1 (word overlap), facts (all names/numbers present).
!python -u ml/train.py {ARGS} --run {RUN} --checkpoints {CKPT} --resume {PRETRAIN_ARGS} 2>&1 | tee -a {CKPT}/{RUN}.log


In [ ]:
# 8. Curves — saved as a PNG for the Lab page
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"{CKPT}/{RUN}/metrics.jsonl")]
ft = [r for r in rows if r["stage"] == "finetune"]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot([r["epoch"] for r in ft], [r["train_loss"] for r in ft], label="train")
a.plot([r["epoch"] for r in ft], [r["dev_loss"] for r in ft], label="dev")
a.set(title="Loss (answer tokens)", xlabel="epoch", ylabel="cross-entropy"); a.legend()
ev = [r for r in ft if "dev_em" in r]
for key, label in [("dev_em", "exact match"), ("dev_f1", "token F1"), ("dev_facts", "key facts")]:
    b.plot([r["epoch"] for r in ev], [100 * r.get(key, 0) for r in ev], marker="o", label=label)
b.set(title="Dev answers", xlabel="epoch", ylabel="%", ylim=(0, 100)); b.legend()
plt.tight_layout(); plt.savefig(f"{CKPT}/{RUN}/curves.png", dpi=150); plt.show()


In [ ]:
# 9. v4 vs v6: full pipeline on dev and on the stress set (same code, same questions)
for run in ["colab-v4", RUN]:
    if not os.path.exists(f"{CKPT}/{run}/best.pt"):
        print(f"(no {run}/best.pt, skipped)"); continue
    for split in ["dev", "stress"]:
        print(f"\n══════ {run} · {split} ══════")
        !python ml/eval_pipeline.py --checkpoint {CKPT}/{run}/best.pt --device cuda --split {split} --thresholds 0.10 --out {CKPT}/{run}/pipeline_{split}.jsonl | tail -7


In [ ]:
# 10. Ask it anything — the same pipeline the site will use
!python ml/ask.py --checkpoint {CKPT}/{RUN}/best.pt --device cuda "What did he build at HackMX?" "Did SkySAR win anything?" "Can you tell me about ZapTap?" "What need does Answr address?" "Does he know Python?" "Has he used React Native?" "What's the capital of France?"


In [ ]:
# 11. Download model, metrics, curves and evaluations (v4's evaluations too)
!cd {CKPT} && rm -f /content/{RUN}.zip && zip -qr /content/{RUN}.zip {RUN} {RUN}.log colab-v4/pipeline_dev.jsonl colab-v4/pipeline_stress.jsonl -x "*/last.pt" "*/pretrained.pt" "*/wiki-tokens.pt" "*.tmp"
from google.colab import files
files.download(f"/content/{RUN}.zip")
